In [1]:
from google.colab import files
uploaded = files.upload()   # chọn Appliances.csv và book1-100k.csv

Saving Appliances.csv to Appliances (1).csv
Saving book1-100k.csv to book1-100k (1).csv


In [2]:
import re
import numpy as np
import pandas as pd
from IPython.display import display

APP_PATH  = 'Appliances.csv'
BOOK_PATH = 'book1-100k.csv'

pd.set_option('display.max_colwidth', 70)
pd.set_option('display.width', 200)

def ask(prompt, default, cast=str):
    """Hỏi người dùng nhập; nhấn Enter để lấy giá trị mặc định."""
    s = input(f'{prompt} [mặc định: {default}]: ').strip()
    try:
        return cast(s) if s else default
    except ValueError:
        print('Giá trị không hợp lệ -> dùng mặc định.')
        return default

def minmax(s):
    s = s.astype(float)
    rng = s.max() - s.min()
    return (s - s.min()) / rng if rng > 0 else s * 0.0

def rank_table(d):
    d = d.copy().reset_index(drop=True)
    d.index = d.index + 1
    return d

In [3]:
df = pd.read_csv(APP_PATH, header=None,
                 names=['ItemID', 'UserID', 'Rating', 'Timestamp'],
                 dtype={'ItemID': str, 'UserID': str})

df['Date'] = pd.to_datetime(df['Timestamp'], unit='s')
df['Year'] = df['Date'].dt.year

print(df.shape)
print('Số người dùng:', df.UserID.nunique(), '| Số sản phẩm:', df.ItemID.nunique())
print('Thời gian:', df.Date.min().date(), '->', df.Date.max().date())
display(df.head())

item_stats = df.groupby('ItemID')['Rating'].agg(count='count', mean='mean').reset_index()

# Weighted Rating: WR = v/(v+m)*R + m/(v+m)*C
def weighted_rating(count, mean, m, c):
    return count / (count + m) * mean + m / (count + m) * c

C_GLOBAL = df['Rating'].mean()
M_GLOBAL = item_stats['count'].quantile(0.90)
item_stats['WR'] = weighted_rating(item_stats['count'], item_stats['mean'], M_GLOBAL, C_GLOBAL)
print(f'C = {C_GLOBAL:.3f} | m = {M_GLOBAL:.0f}')

DEMO_USER = df.UserID.value_counts().index[0]
DEMO_NEG_USER = df[df.Rating <= 2].UserID.value_counts().index[0]
print('Người dùng demo:', DEMO_USER, '| demo tiêu cực:', DEMO_NEG_USER)

(602777, 6)
Số người dùng: 515650 | Số sản phẩm: 30252
Thời gian: 2000-10-23 -> 2018-10-04


,ItemID,UserID,Rating,Timestamp,Date,Year
0,1118461304,A3NHUQ33CFH3VM,5.0,1385510400,2013-11-27,2013
1,1118461304,A3SK6VNBQDNBJE,5.0,1383264000,2013-11-01,2013
2,1118461304,A3SOFHUR27FO3K,5.0,1381363200,2013-10-10,2013
3,1118461304,A1HOG1PYCAE157,5.0,1381276800,2013-10-09,2013
4,1118461304,A26JGAM6GZMM4V,5.0,1378512000,2013-09-07,2013


C = 4.274 | m = 32
Người dùng demo: A8WEXFRWX1ZHH | demo tiêu cực: A352S1XWW1BPA1


In [4]:
# Câu 1. Hệ khuyến nghị thiết bị gia dụng được đánh giá nhiều nhất (Most Popular Recommendation)
# Thống kê số lượt đánh giá của từng sản phẩm, đề xuất Top 10.
def most_popular(n=10):
    out = (df.groupby('ItemID')['Rating'].count()
             .sort_values(ascending=False).head(n)
             .rename('SoLuotDanhGia').reset_index())
    return rank_table(out)

most_popular(10)

,ItemID,SoLuotDanhGia
1,B000AST3AK,6510
2,B004UB1O9Q,5702
3,B0014CN8Y8,4048
4,B00KJ07SEM,3200
5,B0045LLC7K,2936
6,B001B35APA,2481
7,B00INXG9MY,2331
8,B00E37TQV0,2314
9,B004INUWX0,2232
10,B01H1R0K68,2208


In [6]:
# Câu 2. Hệ khuyến nghị thiết bị gia dụng có điểm đánh giá cao nhất (Highest Rated Recommendation)
# Rating trung bình, chỉ xét sản phẩm có ít nhất 10 lượt đánh giá, Top 10.
def highest_rated(min_reviews=10, n=10):
    r = item_stats[item_stats['count'] >= min_reviews]
    r = r.sort_values(['mean', 'count'], ascending=False).head(n)
    return rank_table(r[['ItemID', 'count', 'mean']])

highest_rated(min_reviews=10, n=10)

,ItemID,count,mean
1,B00MGMWO18,33,5.0
2,B01BR37BNW,30,5.0
3,B000XSTXKW,28,5.0
4,B0053Y2SY4,28,5.0
5,B01BTY5AR8,25,5.0
6,B00G1968H8,24,5.0
7,B005XB4JVC,23,5.0
8,B00147FJ3G,22,5.0
9,B00OB1YETM,22,5.0
10,B00Y1RD082,22,5.0


In [5]:
# Câu 3. Hệ khuyến nghị thiết bị gia dụng có mức độ hài lòng cao nhất (Customer Satisfaction Recommendation)
# Tích cực = 4-5 sao; tỷ lệ tích cực; ít nhất 20 lượt đánh giá; Top 10.
def most_satisfying(min_reviews=20, n=10):
    g = (df.assign(positive=(df['Rating'] >= 4))
           .groupby('ItemID')
           .agg(count=('Rating', 'size'), positive=('positive', 'sum'))
           .reset_index())
    g['positive_ratio'] = g['positive'] / g['count']
    g = g[g['count'] >= min_reviews].sort_values(['positive_ratio', 'count'], ascending=False).head(n)
    return rank_table(g)

most_satisfying(min_reviews=20, n=10)

,ItemID,count,positive,positive_ratio
1,B00DM8KWUU,95,95,1.0
2,B001EYAU88,81,81,1.0
3,B00DM8J5MQ,68,68,1.0
4,B0189H5EHE,66,66,1.0
5,B00GMJ2C7M,62,62,1.0
6,B00E64WE0K,60,60,1.0
7,B00M75VU8W,56,56,1.0
8,B001DPP650,55,55,1.0
9,B00AUDA192,50,50,1.0
10,B00DM8KH4G,49,49,1.0


In [7]:
# Câu 4. Hệ khuyến nghị theo mức điểm yêu cầu của khách hàng (Rating Preference Recommendation)
# Nhập điểm TB tối thiểu (1-5) và số lượt đánh giá tối thiểu -> Top 10.
def rating_preference(min_avg, min_reviews, n=10):
    if not 1 <= min_avg <= 5:
        raise ValueError('Điểm trung bình tối thiểu phải nằm trong khoảng 1 đến 5')
    r = item_stats[(item_stats['mean'] >= min_avg) & (item_stats['count'] >= min_reviews)]
    if r.empty:
        print('Không có sản phẩm nào thỏa điều kiện.')
        return r
    return rank_table(r.sort_values(['mean', 'count'], ascending=False).head(n)[['ItemID', 'count', 'mean']])

min_avg = ask('Nhập điểm trung bình tối thiểu (1-5)', 4.5, float)
min_rv  = ask('Nhập số lượt đánh giá tối thiểu', 20, int)
rating_preference(min_avg, min_rv)

Nhập điểm trung bình tối thiểu (1-5) [mặc định: 4.5]: 2
Nhập số lượt đánh giá tối thiểu [mặc định: 20]: 20


,ItemID,count,mean
1,B00MGMWO18,33,5.0
2,B01BR37BNW,30,5.0
3,B000XSTXKW,28,5.0
4,B0053Y2SY4,28,5.0
5,B01BTY5AR8,25,5.0
6,B00G1968H8,24,5.0
7,B005XB4JVC,23,5.0
8,B00147FJ3G,22,5.0
9,B00OB1YETM,22,5.0
10,B00Y1RD082,22,5.0


In [8]:
# Câu 5. Hệ khuyến nghị theo khoảng thời gian đánh giá (Time-Range Recommendation)
# Timestamp -> ngày tháng; nhập năm bắt đầu/kết thúc; Top 10 sản phẩm đánh giá tốt nhất.
def time_range_recommend(year_start, year_end, min_reviews=5, n=10):
    sub = df[(df['Year'] >= year_start) & (df['Year'] <= year_end)]
    if sub.empty:
        print('Không có đánh giá nào trong khoảng thời gian này.')
        return sub
    st = sub.groupby('ItemID')['Rating'].agg(count='count', mean='mean').reset_index()
    c, m = sub['Rating'].mean(), st['count'].quantile(0.90)
    st['WR'] = weighted_rating(st['count'], st['mean'], m, c)
    st = st[st['count'] >= min_reviews].sort_values('WR', ascending=False).head(n)
    return rank_table(st)

y0 = ask('Nhập năm bắt đầu', 2014, int)
y1 = ask('Nhập năm kết thúc', 2016, int)
time_range_recommend(y0, y1)

Nhập năm bắt đầu [mặc định: 2014]: 2014
Nhập năm kết thúc [mặc định: 2016]: 2018


,ItemID,count,mean,WR
1,B00DM8J11Q,1043,4.881112,4.864882
2,B01FZJXITU,378,4.883598,4.840730
3,B0053F80JA,1213,4.850783,4.837504
4,B00IVO8478,179,4.921788,4.832622
5,B013J8CLLU,211,4.890995,4.817502
6,B006H7HB7K,440,4.836364,4.802166
7,B001EYAU88,81,4.987654,4.801963
8,B01DJFFT62,289,4.844291,4.793160
9,B01ET6H58U,347,4.835735,4.793151
10,B00C7MWQS8,255,4.850980,4.793045


In [9]:
# Câu 6. Hệ khuyến nghị có chất lượng đánh giá ổn định (Consistent Rating Recommendation)
# Điểm TB, độ lệch chuẩn, số lượt; ưu tiên TB cao, std thấp, ít nhất 20 lượt.
def consistent_rating(min_reviews=20, n=10, lam=1.0):
    st = (df.groupby('ItemID')['Rating']
            .agg(count='count', mean='mean', std='std').reset_index())
    st = st[st['count'] >= min_reviews].copy()
    st['score'] = st['mean'] - lam * st['std']      # công thức xếp hạng
    return rank_table(st.sort_values('score', ascending=False).head(n))

consistent_rating(min_reviews=20, n=10, lam=1.0)

,ItemID,count,mean,std,score
1,B00147FJ3G,22,5.0,0.0,5.0
2,B00G1968H8,24,5.0,0.0,5.0
3,B00AF7V7HI,21,5.0,0.0,5.0
4,B0053Y2SY4,28,5.0,0.0,5.0
5,B00Y1RD082,22,5.0,0.0,5.0
6,B01BTY5AR8,25,5.0,0.0,5.0
7,B01BR37BNW,30,5.0,0.0,5.0
8,B000XSTXKW,28,5.0,0.0,5.0
9,B00OB1YETM,22,5.0,0.0,5.0
10,B005XB4JVC,23,5.0,0.0,5.0


In [10]:
# Câu 7. Hệ khuyến nghị phát hiện thiết bị tiềm năng ít phổ biến (Hidden Gems Recommendation)
# TB >= 4 sao, 5 <= số lượt <= 30, xếp bằng Weighted Rating, Top 10.
def hidden_gems(n=10):
    g = item_stats[(item_stats['mean'] >= 4) & item_stats['count'].between(5, 30)]
    return rank_table(g.sort_values('WR', ascending=False).head(n))

hidden_gems()

,ItemID,count,mean,WR
1,B01BR37BNW,30,5.000000,4.625398
2,B0053Y2SY4,28,5.000000,4.612912
3,B000XSTXKW,28,5.000000,4.612912
4,B00DM8KL2Y,30,4.966667,4.609269
5,B00LGUB7QY,29,4.965517,4.602864
6,B01BTY5AR8,25,5.000000,4.592538
7,B00UW7AZUG,27,4.962963,4.589402
8,B001GDGHKC,27,4.962963,4.589402
9,B005AR9K0E,29,4.931034,4.586470
10,B00JVZPSMU,29,4.931034,4.586470


In [11]:
# Câu 8. Hệ khuyến nghị sản phẩm chưa được người dùng đánh giá (Unseen Item Recommendation)
# Nhập UserID, loại sản phẩm đã đánh giá, Top 10 theo Weighted Rating.
def seen_items(user_id):
    return set(df.loc[df['UserID'] == user_id, 'ItemID'])

def unseen_recommend(user_id, n=10):
    seen = seen_items(user_id)
    if not seen:
        print('Không tìm thấy UserID này trong dữ liệu.')
        return None
    print(f'Người dùng {user_id} đã đánh giá {len(seen)} sản phẩm -> loại khỏi tập ứng viên.')
    cand = item_stats[~item_stats['ItemID'].isin(seen)]
    return rank_table(cand.sort_values('WR', ascending=False).head(n))

uid = ask('Nhập UserID', DEMO_USER)
unseen_recommend(uid)

Nhập UserID [mặc định: A8WEXFRWX1ZHH]: dfadsasd
Không tìm thấy UserID này trong dữ liệu.


In [12]:
# Câu 9. Hệ khuyến nghị dựa trên lịch sử đánh giá tích cực (Positive Feedback Recommendation)
# Sản phẩm user chấm 4-5 sao -> tìm người dùng cùng sở thích -> đề xuất sản phẩm họ đánh giá tích cực.
pos = df[df['Rating'] >= 4][['UserID', 'ItemID', 'Rating', 'Timestamp']]

def positive_cf_scores(user_id):
    my_items = set(pos.loc[pos['UserID'] == user_id, 'ItemID'])
    if not my_items:
        return pd.Series(dtype=float), my_items
    peers = (pos[pos['ItemID'].isin(my_items) & (pos['UserID'] != user_id)]
             .groupby('UserID').size().rename('shared').reset_index())
    cand = pos[pos['UserID'].isin(peers['UserID']) & ~pos['ItemID'].isin(seen_items(user_id))]
    cand = cand.merge(peers, on='UserID')
    return cand.groupby('ItemID')['shared'].sum().sort_values(ascending=False), my_items

def positive_feedback_recommend(user_id, n=10):
    scores, my_items = positive_cf_scores(user_id)
    if not my_items:
        print('Người dùng này chưa có đánh giá 4-5 sao.')
        return None
    print(f'Người dùng {user_id} yêu thích {len(my_items)} sản phẩm (4-5 sao).')
    out = scores.head(n).rename('score').reset_index()
    out = out.merge(item_stats[['ItemID', 'count', 'mean']], on='ItemID')
    return rank_table(out)

uid = ask('Nhập UserID', DEMO_USER)
positive_feedback_recommend(uid)

Nhập UserID [mặc định: A8WEXFRWX1ZHH]: ádasd
Người dùng này chưa có đánh giá 4-5 sao.


In [13]:
# Câu 10. Hệ khuyến nghị dựa trên lịch sử đánh giá tiêu cực (Negative Feedback Recommendation)
# Sản phẩm user chấm 1-2 sao -> người dùng cùng phản hồi tiêu cực -> giảm ưu tiên sản phẩm nhóm này hay chấm thấp.
def negative_feedback_recommend(user_id, n=10):
    neg_all = df[df['Rating'] <= 2]
    my_neg = set(neg_all.loc[neg_all['UserID'] == user_id, 'ItemID'])
    if not my_neg:
        print('Người dùng này chưa có đánh giá 1-2 sao.')
        return None
    print(f'Người dùng {user_id} đã chấm 1-2 sao cho {len(my_neg)} sản phẩm.')
    peers = neg_all[neg_all['ItemID'].isin(my_neg) & (neg_all['UserID'] != user_id)]['UserID'].unique()
    print('Số người dùng có cùng phản hồi tiêu cực:', len(peers))
    seen = seen_items(user_id)
    peer_rt = df[df['UserID'].isin(peers) & ~df['ItemID'].isin(seen)]
    pen = (peer_rt.assign(low=(peer_rt['Rating'] <= 2))
                  .groupby('ItemID').agg(n_peer=('Rating', 'size'), n_low=('low', 'sum')).reset_index())
    pen['neg_ratio'] = pen['n_low'] / pen['n_peer']
    cand = item_stats[~item_stats['ItemID'].isin(seen)].merge(pen, on='ItemID', how='left').fillna({'neg_ratio': 0, 'n_peer': 0, 'n_low': 0})
    cand['final_score'] = cand['WR'] * (1 - cand['neg_ratio'])      # giảm ưu tiên
    out = cand.sort_values('final_score', ascending=False).head(n)
    print('\nTop khuyến nghị sau khi giảm ưu tiên:')
    display(rank_table(out[['ItemID', 'count', 'mean', 'WR', 'n_peer', 'neg_ratio', 'final_score']]))
    print('Các sản phẩm bị GIẢM ƯU TIÊN nhiều nhất:')
    return rank_table(cand[cand['n_peer'] >= 2].sort_values(['neg_ratio', 'n_peer'], ascending=False)
                      .head(n)[['ItemID', 'count', 'mean', 'WR', 'n_peer', 'neg_ratio', 'final_score']])

uid = ask('Nhập UserID', DEMO_NEG_USER)
negative_feedback_recommend(uid)

Nhập UserID [mặc định: A352S1XWW1BPA1]: ádasd
Người dùng này chưa có đánh giá 1-2 sao.


In [40]:
# Câu 11. Sách tương tự đa thuộc tính (Multi-Attribute Content-Based Recommendation)
# Authors, Publisher, Language, pagesNumber, PublishYear; trọng số từng đặc trưng; Top 10.
WEIGHTS = {'Authors': 0.30, 'Publisher': 0.20, 'Language': 0.10, 'pagesNumber': 0.20, 'PublishYear': 0.20}
PAGE_SCALE, YEAR_SCALE = 500, 50

def content_similarity(book_id, weights=WEIGHTS):
    q = bk[bk['Id'] == book_id].iloc[0]
    s = weights['Authors'] * (bk['Authors'] == q['Authors'])
    s = s + weights['Publisher'] * ((bk['Publisher'] == q['Publisher']) & bk['Publisher'].notna())
    s = s + weights['Language'] * ((bk['Language'] == q['Language']) & bk['Language'].notna())
    s = s + weights['pagesNumber'] * (1 - (bk['pagesNumber'] - q['pagesNumber']).abs().div(PAGE_SCALE).clip(0, 1))
    s = s + weights['PublishYear'] * (1 - (bk['PublishYear'] - q['PublishYear']).abs().div(YEAR_SCALE).clip(0, 1))
    return s / sum(weights.values())

def multi_attribute_similar(book_id, n=10, weights=WEIGHTS):
    q = bk[bk['Id'] == book_id]
    if q.empty:
        print('Không tìm thấy Id này.'); return None
    print('Sách đầu vào:', q.iloc[0]['Name'])
    c = bk.assign(similarity=content_similarity(book_id, weights))
    c = c[c['Id'] != book_id].sort_values(['similarity', 'WR'], ascending=False).head(n)
    return rank_table(c[['Id', 'Name', 'Authors', 'Publisher', 'Language', 'pagesNumber', 'PublishYear', 'similarity']])

bid = ask('Nhập mã sách Id', 3, int)
multi_attribute_similar(bid)

Nhập mã sách Id [mặc định: 3]: 4
Sách đầu vào: Harry Potter and the Chamber of Secrets (Harry Potter, #2)


,Id,Name,Authors,Publisher,Language,pagesNumber,PublishYear,similarity
1,6,"Harry Potter and the Goblet of Fire (Harry Potter, #4)",J.K. Rowling,Scholastic,eng,734,2002,0.8432
2,8,"Harry Potter Boxed Set, Books 1-5 (Harry Potter, #1-5)",J.K. Rowling,Scholastic,eng,2690,2004,0.7960
3,10,"Harry Potter Collection (Harry Potter, #1-6)",J.K. Rowling,Scholastic,eng,3342,2005,0.7920
4,3,"Harry Potter and the Sorcerer's Stone (Harry Potter, #1)",J.K. Rowling,Scholastic Inc,eng,309,2003,0.7828
5,77523,Harry Potter and the Sorcerer's Stone,J.K. Rowling,Arthur A. Levine Books / Scholastic Inc.,eng,309,1998,0.7628
6,5,"Harry Potter and the Prisoner of Azkaban (Harry Potter, #3)",J.K. Rowling,Scholastic Inc.,eng,435,2004,0.7628
7,2002,Harry Potter Schoolbooks Box Set: Two Classic Books from the Libra...,J.K. Rowling,Arthur A. Levine,eng,240,2001,0.7472
8,4255,Harry Potter and the Philosopher's Stone,J.K. Rowling,Bloomsbury Publishing PLC,eng,223,2004,0.7444
9,47538,"Harry Potter et le Prisonnier d'Azkaban (Harry Potter, #3)",J.K. Rowling,NaN,fre,349,2005,0.6908
10,93124,"Harry Potter and the Half-Blood Prince (Harry Potter, #6)",J.K. Rowling,Bloomsbury Publishing,eng,607,2005,0.6900


In [15]:
# Câu 12. Hệ khuyến nghị dựa trên người dùng tương đồng (User-Based Collaborative Filtering)
# pivot_table() User-Item trên tập người dùng đủ dữ liệu; cosine; Top 10.
from sklearn.metrics.pairwise import cosine_similarity

ic = df['ItemID'].value_counts()
sub = df[df['ItemID'].isin(ic[ic >= 20].index)]
uc = sub['UserID'].value_counts()
sub = sub[sub['UserID'].isin(uc[uc >= 3].index)]

R = sub.pivot_table(index='UserID', columns='ItemID', values='Rating', aggfunc='mean').astype('float32')
R0 = R.fillna(0)
print('Kích thước ma trận User-Item:', R.shape)
UB_USER = R.notna().sum(axis=1).idxmax()

def ubcf_scores(user_id, k=30, min_neighbors=2):
    if user_id not in R.index:
        return pd.DataFrame(columns=['ItemID', 'pred', 'n_neighbors'])
    sims = cosine_similarity(R0.loc[[user_id]].values, R0.values)[0]
    sims = pd.Series(sims, index=R.index).drop(user_id)
    nb = sims.nlargest(k)
    nb = nb[nb > 0]
    Rn = R.loc[nb.index].values
    w = nb.values[:, None]
    rated = ~np.isnan(Rn)
    num = np.nansum(Rn * w, axis=0)
    den = (rated * w).sum(axis=0)
    cnt = rated.sum(axis=0)
    pred = pd.DataFrame({'ItemID': R.columns, 'pred': np.where(den > 0, num / np.where(den > 0, den, 1), np.nan), 'n_neighbors': cnt})
    pred = pred[~pred['ItemID'].isin(R.columns[R.loc[user_id].notna()])]
    pred = pred[pred['n_neighbors'] >= min_neighbors].dropna()
    return pred

def user_based_cf(user_id, k=30, n=10):
    if user_id not in R.index:
        print('UserID không nằm trong tập con đủ dữ liệu. Ví dụ UserID hợp lệ:', UB_USER)
        return None
    pred = ubcf_scores(user_id, k)
    return rank_table(pred.sort_values(['pred', 'n_neighbors'], ascending=False).head(n))

uid = ask('Nhập UserID', UB_USER)
user_based_cf(uid)

Kích thước ma trận User-Item: (9576, 3679)
Nhập UserID [mặc định: A3GP2HMB2AFOKF]: ád
UserID không nằm trong tập con đủ dữ liệu. Ví dụ UserID hợp lệ: A3GP2HMB2AFOKF


In [16]:
# Câu 13. Hệ khuyến nghị dựa trên sản phẩm tương đồng (Item-Based Collaborative Filtering)
# Nhập ItemID; Pearson hoặc Cosine; Top 10 sản phẩm tương đồng.
def item_based_cf(item_id, method='cosine', min_common=3, n=10):
    if item_id not in R.columns:
        print('ItemID không nằm trong ma trận (cần >= 20 đánh giá). Ví dụ:', R.notna().sum().idxmax())
        return None
    if method == 'pearson':
        M = R.sub(R.mean(axis=0), axis=1).fillna(0)
    else:
        M = R0
    sims = cosine_similarity(M[[item_id]].T.values, M.T.values)[0]
    nb = R.notna().values.astype('float32')
    common = nb.T @ nb[:, R.columns.get_loc(item_id)]
    out = pd.DataFrame({'ItemID': R.columns, 'similarity': sims, 'common_users': common})
    out = out[(out['ItemID'] != item_id) & (out['common_users'] >= min_common)]
    return rank_table(out.sort_values('similarity', ascending=False).head(n))

iid = ask('Nhập ItemID', R.notna().sum().idxmax())
item_based_cf(iid, method='cosine')

Nhập ItemID [mặc định: B004UB1O9Q]: ád
ItemID không nằm trong ma trận (cần >= 20 đánh giá). Ví dụ: B004UB1O9Q


In [17]:
# Câu 14. Hệ khuyến nghị sản phẩm thường được đánh giá tích cực cùng nhau (Item Co-Occurrence Recommendation)
# merge() tìm cặp sản phẩm cùng được 1 người chấm 4-5 sao; đếm đồng xuất hiện; Top 10.
pos_u = pos[['UserID', 'ItemID']].drop_duplicates()
multi = pos_u['UserID'].value_counts()
pp = pos_u[pos_u['UserID'].isin(multi[(multi >= 2) & (multi <= 200)].index)]
pairs = pp.merge(pp, on='UserID', suffixes=('_a', '_b'))
pairs = pairs[pairs['ItemID_a'] < pairs['ItemID_b']]
pair_counts = pairs.groupby(['ItemID_a', 'ItemID_b']).size().rename('co_count').reset_index()
print('Số cặp sản phẩm đồng xuất hiện:', len(pair_counts))
display(pair_counts.sort_values('co_count', ascending=False).head(5))

def co_occurrence_recommend(item_id, n=10):
    a = pair_counts[pair_counts['ItemID_a'] == item_id][['ItemID_b', 'co_count']].rename(columns={'ItemID_b': 'ItemID'})
    b = pair_counts[pair_counts['ItemID_b'] == item_id][['ItemID_a', 'co_count']].rename(columns={'ItemID_a': 'ItemID'})
    out = pd.concat([a, b]).sort_values('co_count', ascending=False).head(n)
    if out.empty:
        print('Không có sản phẩm đồng xuất hiện với ItemID này.')
        return None
    return rank_table(out.merge(item_stats[['ItemID', 'count', 'mean']], on='ItemID'))

top_item = pair_counts.groupby('ItemID_a')['co_count'].sum().idxmax()
iid = ask('Nhập ItemID', top_item)
co_occurrence_recommend(iid)

Số cặp sản phẩm đồng xuất hiện: 77944


,ItemID_a,ItemID_b,co_count
27787,B004UB1O9Q,B00AHR3IG4,1462
1699,B0002GD194,B00312T5GE,344
76236,B0117DBFNO,B01E85LYNO,310
62038,B00E37TR5U,B00HAJ8IVW,250
73979,B00PQ2GJNY,B00PQ4RM7O,238


Nhập ItemID [mặc định: B004UB1O9Q]: fghjkl;
Không có sản phẩm đồng xuất hiện với ItemID này.


In [18]:
# Câu 15. Hệ khuyến nghị dựa trên cộng đồng người dùng cùng sở thích (Community-Based Recommendation)
# Nhóm người dùng có tập sản phẩm tích cực giống nhau; cộng đồng gần nhất; Top 10.
user_set = pos_u.groupby('UserID')['ItemID'].agg(lambda s: tuple(sorted(s)))
user_size = user_set.map(len)
communities = (user_set.reset_index().rename(columns={'ItemID': 'items'})
               .groupby('items').agg(n_users=('UserID', 'size')).reset_index())
communities['n_items'] = communities['items'].map(len)
print('Số cộng đồng:', len(communities))
display(communities[communities['n_items'] >= 2].sort_values('n_users', ascending=False).head(5))

def community_recommend(user_id, n=10, top_comm=30):
    if user_id not in user_set.index:
        print('Người dùng này chưa có đánh giá tích cực.')
        return None
    my = set(user_set[user_id])
    peers = pos_u.merge(pd.DataFrame({'ItemID': list(my)}), on='ItemID')
    peers = peers[peers['UserID'] != user_id]
    shared = peers.groupby('UserID').size()
    jac = (shared / (len(my) + user_size.loc[shared.index] - shared)).rename('jaccard')
    near = pd.DataFrame({'jaccard': jac, 'items': user_set.loc[jac.index]})
    comm = near.groupby('items').agg(jaccard=('jaccard', 'first'), n_users=('jaccard', 'size')).reset_index()
    comm = comm.sort_values(['jaccard', 'n_users'], ascending=False).head(top_comm)
    comm['w'] = comm['jaccard'] * comm['n_users']
    print('Cộng đồng gần nhất: Jaccard =', round(comm.iloc[0]['jaccard'], 3), '| số người =', comm.iloc[0]['n_users'])
    ex = comm.explode('items').rename(columns={'items': 'ItemID'})
    ex = ex[~ex['ItemID'].isin(seen_items(user_id))]
    out = ex.groupby('ItemID')['w'].sum().sort_values(ascending=False).head(n).rename('score').reset_index()
    return rank_table(out.merge(item_stats[['ItemID', 'count', 'mean']], on='ItemID'))

uid = ask('Nhập UserID', DEMO_USER)
community_recommend(uid)

Số cộng đồng: 59899


,items,n_users,n_items
19961,"(B004UB1O9Q, B00AHR3IG4)",1261,2
55163,"(B0117DBFNO, B01E85LYNO)",267,2
1298,"(B0002GD194, B00312T5GE)",264,2
50996,"(B00PQ2GJNY, B00PQ4RM7O)",217,2
38952,"(B00E37TR5U, B00HAJ8IVW)",184,2


Nhập UserID [mặc định: A8WEXFRWX1ZHH]: fghjkl;
Người dùng này chưa có đánh giá tích cực.


In [19]:
# Câu 16. Hệ khuyến nghị thiết bị gia dụng nổi bật gần đây (Recent Trending Recommendation)
# Lọc 12 tháng cuối của dữ liệu; số lượt + điểm TB; Top 10.
def recent_trending(months=12, n=10):
    latest = df['Date'].max()
    start = latest - pd.DateOffset(months=months)
    print(f'Thời điểm mới nhất: {latest.date()} | lọc từ {start.date()}')
    sub = df[df['Date'] > start]
    st = sub.groupby('ItemID')['Rating'].agg(count='count', mean='mean').reset_index()
    st['WR'] = weighted_rating(st['count'], st['mean'], st['count'].quantile(0.90), sub['Rating'].mean())
    return rank_table(st.sort_values(['WR', 'count'], ascending=False).head(n))

recent_trending()

Thời điểm mới nhất: 2018-10-04 | lọc từ 2017-10-04


,ItemID,count,mean,WR
1,B00LHR20E4,74,4.918919,4.814144
2,B01FZJXITU,191,4.853403,4.812895
3,B00DM8J11Q,86,4.895349,4.806369
4,B01DJFFT62,249,4.831325,4.800880
5,B00NLWBWEM,54,4.925926,4.788515
6,B017Q12A62,91,4.868132,4.786832
7,B0189H5EHE,38,4.973684,4.780990
8,B01FT49YW6,63,4.888889,4.774086
9,B00DM8JA5S,45,4.933333,4.772940
10,B00M75VU8W,36,4.972222,4.772051


In [20]:
# Câu 17. Hệ khuyến nghị lai kết hợp nhiều phương pháp (Hybrid Recommendation)
# Weighted Rating + User-Based CF + Time-Decay; chuẩn hóa; điểm tổng hợp; Top 10 cho từng người dùng.
def time_decay_scores(half_life=365, shrink_k=5):
    ref = df['Date'].max()
    age = (ref - df['Date']).dt.days
    w = 0.5 ** (age / half_life)
    t = df.assign(w=w, wr=w * df['Rating']).groupby('ItemID').agg(sw=('w', 'sum'), swr=('wr', 'sum'))
    t['TD'] = (t['swr'] + shrink_k * C_GLOBAL) / (t['sw'] + shrink_k)
    return t['TD'].reset_index()

td_table = time_decay_scores()

def hybrid_recommend(user_id, w_wr=0.3, w_ub=0.4, w_td=0.3, n=10):
    s = w_wr + w_ub + w_td
    w_wr, w_ub, w_td = w_wr / s, w_ub / s, w_td / s
    seen = seen_items(user_id)
    if not seen:
        print('Không tìm thấy UserID này.')
        return None
    cand = item_stats[~item_stats['ItemID'].isin(seen)][['ItemID', 'count', 'mean', 'WR']]
    cand = cand.merge(td_table, on='ItemID', how='left')
    ub = ubcf_scores(user_id)[['ItemID', 'pred']] if user_id in R.index else pd.DataFrame(columns=['ItemID', 'pred'])
    if ub.empty:
        print('(Người dùng không có trong ma trận User-Item -> thành phần UBCF = 0)')
    cand = cand.merge(ub, on='ItemID', how='left')
    cand['pred'] = cand['pred'].fillna(0)
    cand['WR_n'], cand['UB_n'], cand['TD_n'] = minmax(cand['WR']), minmax(cand['pred']), minmax(cand['TD'])
    cand['hybrid'] = w_wr * cand['WR_n'] + w_ub * cand['UB_n'] + w_td * cand['TD_n']
    return rank_table(cand.sort_values('hybrid', ascending=False).head(n)
                      [['ItemID', 'count', 'mean', 'WR_n', 'UB_n', 'TD_n', 'hybrid']])

uid = ask('Nhập UserID', UB_USER)
hybrid_recommend(uid)

Nhập UserID [mặc định: A3GP2HMB2AFOKF]: dfghjkl;'
Không tìm thấy UserID này.


In [21]:
# Câu 18. Đánh giá bằng Precision@K, Recall@K, Hit Rate@K (Recommendation Evaluation)
# Popularity, User-Based CF, Item-Based CF; chia train/test theo thời gian; so sánh.
from scipy import sparse
from sklearn.preprocessing import normalize

K = 10
d = df.sort_values('Timestamp').drop_duplicates(['UserID', 'ItemID'], keep='last')
cut = d['Timestamp'].quantile(0.80)
train, test = d[d['Timestamp'] <= cut], d[d['Timestamp'] > cut]
print('Train:', len(train), '| Test:', len(test), '| Mốc cắt:', pd.to_datetime(cut, unit='s').date())

uid_cat = train['UserID'].astype('category'); iid_cat = train['ItemID'].astype('category')
user_idx = {u: i for i, u in enumerate(uid_cat.cat.categories)}
item_idx = {i: j for j, i in enumerate(iid_cat.cat.categories)}
item_names = np.array(iid_cat.cat.categories)
X = sparse.csr_matrix((train['Rating'].values, (uid_cat.cat.codes, iid_cat.cat.codes)),
                      shape=(len(user_idx), len(item_idx)))
Xu = normalize(X)
Xi = normalize(X.T.tocsr())
pop_score = np.asarray((X > 0).sum(axis=0)).ravel().astype(float)

test_pos = test[(test['Rating'] >= 4) & test['ItemID'].isin(item_idx) & test['UserID'].isin(user_idx)]
relevant = test_pos.groupby('UserID')['ItemID'].agg(set)
rng = np.random.RandomState(42)
eval_users = rng.choice(relevant.index, size=min(500, len(relevant)), replace=False)
print('Số người dùng đánh giá:', len(eval_users))

def top_k(scores, u):
    scores = scores.copy()
    scores[X[u].indices] = -np.inf
    idx = np.argpartition(-scores, K)[:K]
    return item_names[idx[np.argsort(-scores[idx])]]

def rec_popularity(u): return top_k(pop_score, u)

def rec_ubcf(u, k=50):
    sims = (Xu[u] @ Xu.T).toarray().ravel(); sims[u] = 0
    nb = np.argpartition(-sims, k)[:k]
    return top_k((sparse.csr_matrix(sims[nb]) @ X[nb]).toarray().ravel(), u)

def rec_ibcf(u):
    items = X[u].indices; r = X[u].data
    S = Xi[items] @ Xi.T
    return top_k((sparse.csr_matrix(r) @ S).toarray().ravel(), u)

def evaluate(rec_fn):
    P, Rc, H = [], [], []
    for u in eval_users:
        rel = relevant[u]
        rec = set(rec_fn(user_idx[u]))
        hit = len(rec & rel)
        P.append(hit / K); Rc.append(hit / len(rel)); H.append(1.0 if hit > 0 else 0.0)
    return {'Precision@10': np.mean(P), 'Recall@10': np.mean(Rc), 'HitRate@10': np.mean(H)}

res = pd.DataFrame({'Popularity': evaluate(rec_popularity),
                    'User-Based CF': evaluate(rec_ubcf),
                    'Item-Based CF': evaluate(rec_ibcf)}).T
display(res.round(4))
print('=> Phương pháp tốt nhất theo Hit Rate@10:', res['HitRate@10'].idxmax())

Train: 472801 | Test: 118043 | Mốc cắt: 2017-08-26
Số người dùng đánh giá: 500


,Precision@10,Recall@10,HitRate@10
Popularity,0.0016,0.0140,0.016
User-Based CF,0.0010,0.0090,0.010
Item-Based CF,0.0034,0.0305,0.034


=> Phương pháp tốt nhất theo Hit Rate@10: Item-Based CF


In [22]:
bk = pd.read_csv(BOOK_PATH)

for i in range(1, 6):
    bk[f'n{i}'] = bk[f'RatingDist{i}'].str.split(':').str[1].astype(int)
bk['total'] = bk['RatingDistTotal'].str.split(':').str[1].astype(int)

print(bk.shape)
display(bk[['Id', 'Name', 'Authors', 'Publisher', 'Language', 'pagesNumber', 'PublishYear', 'Rating', 'total', 'CountsOfReview']].head())

def weighted_rating(count, mean, m, c):
    return count / (count + m) * mean + m / (count + m) * c

C_BK = bk.loc[bk['total'] > 0, 'Rating'].mean()
M_BK = bk['total'].quantile(0.90)
bk['WR'] = weighted_rating(bk['total'], bk['Rating'], M_BK, C_BK)
print(f'C = {C_BK:.3f} | m = {M_BK:.0f}')

SHOW = ['Id', 'Name', 'Authors', 'Publisher', 'Rating', 'total']

(58292, 24)


,Id,Name,Authors,Publisher,Language,pagesNumber,PublishYear,Rating,total,CountsOfReview
0,1,"Harry Potter and the Half-Blood Prince (Harry Potter, #6)",J.K. Rowling,Scholastic Inc.,eng,652,2006,4.57,2298124,28062
1,2,"Harry Potter and the Order of the Phoenix (Harry Potter, #5)",J.K. Rowling,Scholastic Inc.,eng,870,2004,4.50,2358637,29770
2,3,"Harry Potter and the Sorcerer's Stone (Harry Potter, #1)",J.K. Rowling,Scholastic Inc,eng,309,2003,4.47,6587388,75911
3,4,"Harry Potter and the Chamber of Secrets (Harry Potter, #2)",J.K. Rowling,Scholastic,eng,352,2003,4.42,2560657,244
4,5,"Harry Potter and the Prisoner of Azkaban (Harry Potter, #3)",J.K. Rowling,Scholastic Inc.,eng,435,2004,4.57,2610317,37093


C = 3.892 | m = 12461


In [23]:
# Câu 1. Top 10 cuốn sách có điểm đánh giá cao nhất (Highest Rated Books Recommendation)
# Sắp xếp theo Rating giảm dần; chỉ xét sách có ít nhất 100 lượt đánh giá trong RatingDistTotal.
def highest_rated_books(min_reviews=100, n=10):
    r = bk[bk['total'] >= min_reviews].sort_values(['Rating', 'total'], ascending=False).head(n)
    return rank_table(r[SHOW])

highest_rated_books()

,Id,Name,Authors,Publisher,Rating,total
1,12278,The Grove Centenary Editions of Samuel Beckett (4 Volumes),Samuel Beckett,Grove Press,4.88,105
2,24812,The Complete Calvin and Hobbes,Bill Watterson,Andrews McMeel Publishing,4.82,34079
3,60269,The First Folio of Shakespeare,William Shakespeare,W. W. Norton & Company,4.80,241
4,2154,The Oxford Illustrated Jane Austen: 6-Volume Set,Jane Austen,"Oxford University Press, USA",4.80,131
5,8,"Harry Potter Boxed Set, Books 1-5 (Harry Potter, #1-5)",J.K. Rowling,Scholastic,4.78,43968
6,95602,Mark of the Lion Trilogy,Francine Rivers,Tyndale House,4.77,11152
7,24814,"It's a Magical World (Calvin and Hobbes, #11)",Bill Watterson,Andrews McMeel Publishing,4.76,25429
8,54741,Toda Mafalda,Quino,Ediciones de la Flor,4.76,7613
9,15026,In Conversation with God,Francisco Fernández-Carvajal,Scepter Publishers,4.76,126
10,70489,There's Treasure Everywhere (Calvin and Hobbes #10),Bill Watterson,Andrews McMeel Publishing,4.75,19435


In [24]:
# Câu 2. Top 10 cuốn sách phổ biến nhất (Most Popular Books Recommendation)
# Chuyển RatingDistTotal từ chuỗi về số nguyên; sắp xếp; Top 10.
print(bk[['RatingDistTotal', 'total']].head(3))
print('Kiểu dữ liệu cột total:', bk['total'].dtype)

def most_popular_books(n=10):
    return rank_table(bk.sort_values('total', ascending=False).head(n)[SHOW])

most_popular_books()

  RatingDistTotal    total
0   total:2298124  2298124
1   total:2358637  2358637
2   total:6587388  6587388
Kiểu dữ liệu cột total: int64


,Id,Name,Authors,Publisher,Rating,total
1,77523,Harry Potter and the Sorcerer's Stone,J.K. Rowling,Arthur A. Levine Books / Scholastic Inc.,4.47,6622480
2,63637,Harry Potter agus an Órchloch,J.K. Rowling,Bloomsbury Children's Books,4.47,6601642
3,49772,"Harri Potter a Maen yr Athronydd (Harry Potter, #1)",J.K. Rowling,Bloomsbury Childrens,4.47,6599670
4,49773,Harry Potter e la Pietra Filosofale,J.K. Rowling,Salani,4.47,6599670
5,47535,Ἄρειος Ποτὴρ καὶ ἡ τοῦ φιλοσόφου λίθος (Ἅρειος Ποτῆρ #1),J.K. Rowling,Bloomsbury,4.47,6599481
6,47522,"Harry Potter und der Stein der Weisen (Harry Potter, #1)",J.K. Rowling,Carlsen Verlag GmbH,4.47,6599481
7,43504,"Harry Potter and the Philosopher's Stone (Harry Potter, #1)",J.K. Rowling,Bloomsbury USA Childrens,4.47,6599146
8,15880,Harrius Potter et Philosophi Lapis,J.K. Rowling,Bloomsbury,4.47,6589476
9,4255,Harry Potter and the Philosopher's Stone,J.K. Rowling,Bloomsbury Publishing PLC,4.47,6588013
10,3357,"Harry Potter Y La Piedra Filosofal (Harry Potter, #1)",J.K. Rowling,Turtleback Books,4.47,6587900


In [25]:
# Câu 3. Sách có tỷ lệ đánh giá tích cực cao nhất (Positive Rating Recommendation)
# Dùng RatingDist4, RatingDist5, RatingDistTotal; ít nhất 100 lượt; Top 10.
def positive_rating_books(min_reviews=100, n=10):
    t = bk[bk['total'] >= min_reviews].copy()
    t['positive_ratio'] = (t['n4'] + t['n5']) / t['total']
    t = t.sort_values(['positive_ratio', 'total'], ascending=False).head(n)
    return rank_table(t[SHOW + ['positive_ratio']])

positive_rating_books()

,Id,Name,Authors,Publisher,Rating,total,positive_ratio
1,12278,The Grove Centenary Editions of Samuel Beckett (4 Volumes),Samuel Beckett,Grove Press,4.88,105,0.980952
2,2154,The Oxford Illustrated Jane Austen: 6-Volume Set,Jane Austen,"Oxford University Press, USA",4.80,131,0.977099
3,6550,Early Color,Saul Leiter,Steidl,4.73,158,0.974684
4,60441,Hand to Earth,Andy Goldsworthy,Harry N. Abrams,4.67,128,0.968750
5,60446,Hand to Earth Andy Goldsworth Scuplture 1976-1990,Terry Friedman,ABRAMS,4.67,128,0.968750
6,24812,The Complete Calvin and Hobbes,Bill Watterson,Andrews McMeel Publishing,4.82,34079,0.968661
7,70168,Medicinal Plants of the Desert and Canyon West,Michael Moore,Museum of New Mexico Press,4.62,110,0.963636
8,60269,The First Folio of Shakespeare,William Shakespeare,W. W. Norton & Company,4.80,241,0.962656
9,24814,"It's a Magical World (Calvin and Hobbes, #11)",Bill Watterson,Andrews McMeel Publishing,4.76,25429,0.960557
10,26805,The Sibley Field Guide to Birds of Western North America,David Allen Sibley,Alfred A. Knopf,4.69,826,0.960048


In [26]:
# Câu 4. Hệ khuyến nghị sách theo tác giả yêu thích (Author-Based Recommendation)
# Nhập tên tác giả; sắp xếp theo Weighted Rating; Top 10.
def author_recommend(author, n=10):
    r = bk[bk['Authors'].str.contains(author, case=False, na=False, regex=False)]
    if r.empty:
        print('Không tìm thấy tác giả này.')
        return None
    print(f'Tìm thấy {len(r)} cuốn sách của "{author}"')
    return rank_table(r.sort_values('WR', ascending=False).head(n)[SHOW + ['WR']])

author = ask('Nhập tên tác giả', 'J.K. Rowling')
author_recommend(author)

Nhập tên tác giả [mặc định: J.K. Rowling]: jasd
Không tìm thấy tác giả này.


In [27]:
# Câu 5. Hệ khuyến nghị sách theo nhà xuất bản (Publisher-Based Recommendation)
# Nhập tên NXB; Top 10 điểm cao nhất, số lượt đánh giá đạt ngưỡng tối thiểu.
def publisher_recommend(publisher, min_reviews=100, n=10):
    r = bk[bk['Publisher'].str.contains(publisher, case=False, na=False, regex=False)]
    if r.empty:
        print('Không tìm thấy nhà xuất bản này.')
        return None
    print(f'NXB "{publisher}" có {len(r)} cuốn; {(r.total >= min_reviews).sum()} cuốn đạt ngưỡng {min_reviews} lượt.')
    r = r[r['total'] >= min_reviews].sort_values(['Rating', 'total'], ascending=False).head(n)
    return rank_table(r[SHOW])

publisher = ask('Nhập tên nhà xuất bản', 'Penguin Books')
min_rv = ask('Số lượt đánh giá tối thiểu', 100, int)
publisher_recommend(publisher, min_rv)

Nhập tên nhà xuất bản [mặc định: Penguin Books]: ád
Số lượt đánh giá tối thiểu [mặc định: 100]: ádff
Giá trị không hợp lệ -> dùng mặc định.
Không tìm thấy nhà xuất bản này.


In [ ]:
# Câu 5. Hệ khuyến nghị sách theo nhà xuất bản (Publisher-Based Recommendation)
# Nhập tên NXB; Top 10 điểm cao nhất, số lượt đánh giá đạt ngưỡng tối thiểu.
def publisher_recommend(publisher, min_reviews=100, n=10):
    r = bk[bk['Publisher'].str.contains(publisher, case=False, na=False, regex=False)]
    if r.empty:
        print('Không tìm thấy nhà xuất bản này.')
        return None
    print(f'NXB "{publisher}" có {len(r)} cuốn; {(r.total >= min_reviews).sum()} cuốn đạt ngưỡng {min_reviews} lượt.')
    r = r[r['total'] >= min_reviews].sort_values(['Rating', 'total'], ascending=False).head(n)
    return rank_table(r[SHOW])

publisher = ask('Nhập tên nhà xuất bản', 'Penguin Books')
min_rv = ask('Số lượt đánh giá tối thiểu', 100, int)
publisher_recommend(publisher, min_rv)

In [28]:
# Câu 7. Hệ khuyến nghị sách theo số lượng trang (Book Length Recommendation)
# Nhập số trang tối thiểu/tối đa; lọc theo pagesNumber; ưu tiên điểm đánh giá cao; Top 10.
def book_length_recommend(min_pages, max_pages, n=10):
    r = bk[bk['pagesNumber'].between(min_pages, max_pages)]
    if r.empty:
        print('Không có sách nào trong khoảng số trang này.')
        return None
    return rank_table(r.sort_values('WR', ascending=False).head(n)[SHOW + ['pagesNumber', 'WR']])

p0 = ask('Số trang tối thiểu', 200, int)
p1 = ask('Số trang tối đa', 400, int)
book_length_recommend(p0, p1)

Số trang tối thiểu [mặc định: 200]: 2000
Số trang tối đa [mặc định: 400]: 211
Không có sách nào trong khoảng số trang này.


In [29]:
# Câu 8. Sách tương tự dựa trên tác giả và nhà xuất bản (Author–Publisher Similarity)
# Nhập Id; Top 10 sách khác cùng tác giả hoặc NXB; ưu tiên trùng cả hai.
def author_publisher_similar(book_id, n=10):
    q = bk[bk['Id'] == book_id]
    if q.empty:
        print('Không tìm thấy Id này.'); return None
    q = q.iloc[0]
    print(f'Sách đầu vào: {q.Name} | Tác giả: {q.Authors} | NXB: {q.Publisher}')
    c = bk[bk['Id'] != book_id].copy()
    c['same_author'] = c['Authors'] == q['Authors']
    c['same_publisher'] = (c['Publisher'] == q['Publisher']) & c['Publisher'].notna()
    c = c[c['same_author'] | c['same_publisher']]
    c['n_match'] = c['same_author'].astype(int) + c['same_publisher'].astype(int)
    c = c.sort_values(['n_match', 'WR'], ascending=False).head(n)
    return rank_table(c[SHOW + ['same_author', 'same_publisher', 'n_match']])

bid = ask('Nhập mã sách Id', 3, int)
author_publisher_similar(bid)

Nhập mã sách Id [mặc định: 3]: 3
Sách đầu vào: Harry Potter and the Sorcerer's Stone (Harry Potter, #1) | Tác giả: J.K. Rowling | NXB: Scholastic Inc


,Id,Name,Authors,Publisher,Rating,total,same_author,same_publisher,n_match
1,71643,"Harry Potter and the Deathly Hallows (Harry Potter, #7)",J.K. Rowling,Arthur A. Levine Books,4.62,2661573,True,False,1
2,8,"Harry Potter Boxed Set, Books 1-5 (Harry Potter, #1-5)",J.K. Rowling,Scholastic,4.78,43968,True,False,1
3,72428,Harry Potter et le prisonnier d'Azkaban,J.K. Rowling,Gallimard Jeunesse,4.57,2636159,True,False,1
4,5,"Harry Potter and the Prisoner of Azkaban (Harry Potter, #3)",J.K. Rowling,Scholastic Inc.,4.57,2610317,True,False,1
5,93124,"Harry Potter and the Half-Blood Prince (Harry Potter, #6)",J.K. Rowling,Bloomsbury Publishing,4.57,2306655,True,False,1
6,70367,"Harry Potter and the Half-blood Prince (Harry Potter, #6)",J.K. Rowling,Turtleback Books,4.57,2300327,True,False,1
7,70368,"Harry Potter et le Prince de Sang-Mêlé (Harry Potter, #6)",J.K. Rowling,French & European Pubns,4.57,2300327,True,False,1
8,70371,Гарри Поттер и Принц-полукровка (Гарри Поттер #6),J.K. Rowling,Росмэн,4.57,2300327,True,False,1
9,70373,哈利·波特与混血王子 (哈利·波特 #6),J.K. Rowling,人民文学出版社,4.57,2300327,True,False,1
10,70374,"Harry Potter e il Principe Mezzosangue (Harry Potter, #6)",J.K. Rowling,French & European Pubns,4.57,2300327,True,False,1


In [30]:
# Câu 9. Sách theo mức độ tương đồng của tên sách (Title Similarity Recommendation)
# Tách và chuẩn hóa từ trong Name; Jaccard trên tập từ; Top 10.
STOP = {'the', 'a', 'an', 'of', 'and', 'in', 'to', 'for', 'on', 'with', 'at', 'by', 'from', 'is'}

def name_tokens(s):
    return set(re.findall(r'[a-z0-9]+', str(s).lower())) - STOP

bk['tokens'] = bk['Name'].apply(name_tokens)

def jaccard(a, b):
    u = a | b
    return len(a & b) / len(u) if u else 0.0

def title_similar(book_id, n=10):
    q = bk[bk['Id'] == book_id]
    if q.empty:
        print('Không tìm thấy Id này.'); return None
    q = q.iloc[0]
    print('Sách đầu vào:', q['Name'], '| tập từ:', sorted(q['tokens']))
    sim = bk['tokens'].apply(lambda t: jaccard(t, q['tokens']))
    c = bk.assign(jaccard=sim)
    c = c[c['Id'] != book_id].sort_values(['jaccard', 'WR'], ascending=False).head(n)
    return rank_table(c[['Id', 'Name', 'Authors', 'jaccard']])

bid = ask('Nhập mã sách Id', 3, int)
title_similar(bid)

Nhập mã sách Id [mặc định: 3]: 2
Sách đầu vào: Harry Potter and the Order of the Phoenix (Harry Potter, #5) | tập từ: ['5', 'harry', 'order', 'phoenix', 'potter']


,Id,Name,Authors,jaccard
1,77522,"Harry Potter and the Order of the Phoenix (Harry Potter, #5)",J.K. Rowling,1.000000
2,70356,Harry Potter and the Order of the Phoenix,J.K. Rowling,0.800000
3,70355,"هاري بوتر وجماعة العنقاء (Harry Potter, #5)",J.K. Rowling,0.600000
4,8,"Harry Potter Boxed Set, Books 1-5 (Harry Potter, #1-5)",J.K. Rowling,0.333333
5,49869,"هاري بوتر وسجين أزكابان (Harry Potter, #3)",J.K. Rowling,0.333333
6,49839,"هاري بوتر وكأس النار (Harry Potter, #4)",J.K. Rowling,0.333333
7,70365,Harry Potter und der Orden des Phoenix,J.K. Rowling,0.333333
8,63637,Harry Potter agus an Órchloch,J.K. Rowling,0.285714
9,15190,Looking for God in Harry Potter,John Granger,0.285714
10,72424,"해리 포터와 비밀의 방 2 (Harry Potter #2, part 2 of 2)",J.K. Rowling,0.285714


In [31]:
# Câu 10. Sách theo đặc trưng độ dài và năm xuất bản (Book Feature Similarity)
# Nhập Id; chuẩn hóa pagesNumber, PublishYear; khoảng cách; 10 cuốn gần nhất.
valid = bk[(bk['pagesNumber'] > 0) & (bk['pagesNumber'] <= 5000) & bk['PublishYear'].between(1000, 2026)].copy()
for col in ['pagesNumber', 'PublishYear']:
    valid[col + '_z'] = (valid[col] - valid[col].mean()) / valid[col].std()

def feature_similar(book_id, n=10):
    q = valid[valid['Id'] == book_id]
    if q.empty:
        print('Id không có trong tập sách hợp lệ.'); return None
    q = q.iloc[0]
    valid['distance'] = np.sqrt((valid['pagesNumber_z'] - q['pagesNumber_z']) ** 2 +
                                (valid['PublishYear_z'] - q['PublishYear_z']) ** 2)
    print(f"Sách đầu vào: {q['Name']} | {q['pagesNumber']} trang | năm {q['PublishYear']}")
    r = valid[valid['Id'] != book_id].sort_values(['distance', 'total'], ascending=[True, False]).head(n)
    return rank_table(r[['Id', 'Name', 'pagesNumber', 'PublishYear', 'distance']])

bid = ask('Nhập mã sách Id', 3, int)
feature_similar(bid)

Nhập mã sách Id [mặc định: 3]: 1
Sách đầu vào: Harry Potter and the Half-Blood Prince (Harry Potter, #6) | 652 trang | năm 2006


,Id,Name,pagesNumber,PublishYear,distance
1,70367,"Harry Potter and the Half-blood Prince (Harry Potter, #6)",652,2006,0.000000
2,83518,The Singularity is Near: When Humans Transcend Biology,652,2006,0.000000
3,56382,Anthropology,652,2006,0.000000
4,90360,Ireland,651,2006,0.004621
5,82620,"The Chemistry of Gold Extraction, Second Edition",651,2006,0.004621
6,50022,Introduction to Homeland Security,654,2006,0.009242
7,98568,"Microsoft Visual Basic 2005: Reloaded, Advanced",654,2006,0.009242
8,21062,A Rose for the Crown,650,2006,0.009242
9,89960,America's Constitution: A Biography,655,2006,0.013863
10,97498,From Jim Crow to Civil Rights: The Supreme Court and the Struggle ...,655,2006,0.013863


In [32]:
# Câu 11. Sách tương tự đa thuộc tính (Multi-Attribute Content-Based Recommendation)
# Authors, Publisher, Language, pagesNumber, PublishYear; trọng số từng đặc trưng; Top 10.
WEIGHTS = {'Authors': 0.30, 'Publisher': 0.20, 'Language': 0.10, 'pagesNumber': 0.20, 'PublishYear': 0.20}
PAGE_SCALE, YEAR_SCALE = 500, 50

def content_similarity(book_id, weights=WEIGHTS):
    q = bk[bk['Id'] == book_id].iloc[0]
    s = weights['Authors'] * (bk['Authors'] == q['Authors'])
    s = s + weights['Publisher'] * ((bk['Publisher'] == q['Publisher']) & bk['Publisher'].notna())
    s = s + weights['Language'] * ((bk['Language'] == q['Language']) & bk['Language'].notna())
    s = s + weights['pagesNumber'] * (1 - (bk['pagesNumber'] - q['pagesNumber']).abs().div(PAGE_SCALE).clip(0, 1))
    s = s + weights['PublishYear'] * (1 - (bk['PublishYear'] - q['PublishYear']).abs().div(YEAR_SCALE).clip(0, 1))
    return s / sum(weights.values())

def multi_attribute_similar(book_id, n=10, weights=WEIGHTS):
    q = bk[bk['Id'] == book_id]
    if q.empty:
        print('Không tìm thấy Id này.'); return None
    print('Sách đầu vào:', q.iloc[0]['Name'])
    c = bk.assign(similarity=content_similarity(book_id, weights))
    c = c[c['Id'] != book_id].sort_values(['similarity', 'WR'], ascending=False).head(n)
    return rank_table(c[['Id', 'Name', 'Authors', 'Publisher', 'Language', 'pagesNumber', 'PublishYear', 'similarity']])

bid = ask('Nhập mã sách Id', 3, int)
multi_attribute_similar(bid)

Nhập mã sách Id [mặc định: 3]: 1
Sách đầu vào: Harry Potter and the Half-Blood Prince (Harry Potter, #6)


,Id,Name,Authors,Publisher,Language,pagesNumber,PublishYear,similarity
1,5,"Harry Potter and the Prisoner of Azkaban (Harry Potter, #3)",J.K. Rowling,Scholastic Inc.,eng,435,2004,0.9052
2,2,"Harry Potter and the Order of the Phoenix (Harry Potter, #5)",J.K. Rowling,Scholastic Inc.,eng,870,2004,0.9048
3,70367,"Harry Potter and the Half-blood Prince (Harry Potter, #6)",J.K. Rowling,Turtleback Books,eng,652,2006,0.8000
4,93124,"Harry Potter and the Half-Blood Prince (Harry Potter, #6)",J.K. Rowling,Bloomsbury Publishing,eng,607,2005,0.7780
5,6,"Harry Potter and the Goblet of Fire (Harry Potter, #4)",J.K. Rowling,Scholastic,eng,734,2002,0.7512
6,71643,"Harry Potter and the Deathly Hallows (Harry Potter, #7)",J.K. Rowling,Arthur A. Levine Books,eng,816,2007,0.7304
7,77522,"Harry Potter and the Order of the Phoenix (Harry Potter, #5)",J.K. Rowling,Arthur A. Levine Books / Scholastic Inc.,eng,870,2003,0.7008
8,49839,"هاري بوتر وكأس النار (Harry Potter, #4)",J.K. Rowling,نهضة مصر للطباعة والنشر والتوزيع,ara,658,2006,0.6976
9,70371,Гарри Поттер и Принц-полукровка (Гарри Поттер #6),J.K. Rowling,Росмэн,rus,672,2005,0.6880
10,15872,"Harry Potter y el misterio del príncipe (Harry Potter, #6)",J.K. Rowling,Salamandra,spa,602,2006,0.6800


In [33]:
# Câu 12. Sách có mức đánh giá ổn định (Rating Consistency Recommendation)
# Độ lệch chuẩn từ RatingDist1..5; điểm TB cao, phân tán thấp, đủ lượt đánh giá; Top 10.
def rating_consistency(min_reviews=1000, n=10, lam=1.0):
    t = bk[bk['total'] > 0].copy()
    cnt = t[['n1', 'n2', 'n3', 'n4', 'n5']].values.astype(float)
    stars = np.arange(1, 6)
    mean_d = (cnt * stars).sum(axis=1) / cnt.sum(axis=1)
    t['std_dist'] = np.sqrt((cnt * (stars - mean_d[:, None]) ** 2).sum(axis=1) / cnt.sum(axis=1))
    t = t[t['total'] >= min_reviews]
    t['score'] = t['Rating'] - lam * t['std_dist']
    return rank_table(t.sort_values('score', ascending=False).head(n)[SHOW + ['std_dist', 'score']])

rating_consistency()

,Id,Name,Authors,Publisher,Rating,total,std_dist,score
1,24812,The Complete Calvin and Hobbes,Bill Watterson,Andrews McMeel Publishing,4.82,34079,0.516571,4.303429
2,24814,"It's a Magical World (Calvin and Hobbes, #11)",Bill Watterson,Andrews McMeel Publishing,4.76,25429,0.536946,4.223054
3,95602,Mark of the Lion Trilogy,Francine Rivers,Tyndale House,4.77,11152,0.565832,4.204168
4,70489,There's Treasure Everywhere (Calvin and Hobbes #10),Bill Watterson,Andrews McMeel Publishing,4.75,19435,0.551359,4.198641
5,54741,Toda Mafalda,Quino,Ediciones de la Flor,4.76,7613,0.569663,4.190337
6,59715,The Authoritative Calvin and Hobbes: A Calvin and Hobbes Treasury,Bill Watterson,Sphere,4.74,20481,0.551999,4.188001
7,8,"Harry Potter Boxed Set, Books 1-5 (Harry Potter, #1-5)",J.K. Rowling,Scholastic,4.78,43968,0.610298,4.169702
8,70487,Attack of the Deranged Mutant Killer Monster Snow Goons (Calvin an...,Bill Watterson,Turtleback Books,4.72,13898,0.560952,4.159048
9,24816,Homicidal Psycho Jungle Cat (Calvin and Hobbes #9),Bill Watterson,Andrews McMeel Publishing,4.72,17479,0.562511,4.157489
10,77729,Something Under the Bed Is Drooling,Bill Watterson,Andrews McMeel Publishing,4.71,12567,0.577669,4.132331


In [34]:
# Câu 13. Sách ít nổi tiếng nhưng được đánh giá tốt (Hidden Gems Recommendation)
# Rating >= 4 và 20-200 lượt đánh giá; xếp bằng Weighted Rating; Top 10.
def hidden_gem_books(n=10):
    r = bk[(bk['Rating'] >= 4) & bk['total'].between(20, 200)]
    return rank_table(r.sort_values('WR', ascending=False).head(n)[SHOW + ['WR']])

hidden_gem_books()

,Id,Name,Authors,Publisher,Rating,total,WR
1,41202,Medications and Mothers' Milk 2006 (Medications and Mother's Milk),Thomas W. Hale,Hale Pub.,4.72,179,3.904117
2,83871,Migrations: Humanity in Transition,Sebastião Salgado,Aperture,4.73,169,3.903605
3,84969,Disney's Storybook Collection - Volume 2,Deborah Boone,Disney Press,4.63,181,3.902958
4,83264,The Masnavi: Book Two,Rumi,Oxford University Press,4.57,197,3.902943
5,6550,Early Color,Saul Leiter,Steidl,4.73,158,3.902885
6,2130,Twenty-Four Hours a Day,Anonymous,Hazelden Publishing,4.56,197,3.902787
7,73288,THE Complete Works Of Oswald Chambers,Oswald Chambers,Discovery House Publishers,4.63,178,3.902785
8,53735,"Course of Theoretical Physics: Vol. 2, The Classical Theory of Fields",L.D. Landau,Butterworth-Heinemann,4.58,191,3.902777
9,83002,Silence of the Heart: Dialogues with Robert Adams,Robert Adams,Acropolis Books (GA),4.60,183,3.902638
10,89961,The Heritage Guide to the Constitution,Edwin Meese III,Regnery Publishing,4.54,200,3.902627


In [35]:
# Câu 14. Sách dựa trên mức độ quan tâm của độc giả (Reader Engagement Recommendation)
# CountsOfReview / RatingDistTotal kết hợp Rating; Top 10.
def reader_engagement(min_reviews=100, w_engage=0.5, n=10):
    t = bk[bk['total'] >= min_reviews].copy()
    t['review_ratio'] = t['CountsOfReview'] / t['total']
    cap = t['review_ratio'].quantile(0.99)
    t['engage_n'] = minmax(t['review_ratio'].clip(upper=cap))
    t['rating_n'] = minmax(t['Rating'])
    t['score'] = w_engage * t['engage_n'] + (1 - w_engage) * t['rating_n']
    return rank_table(t.sort_values('score', ascending=False).head(n)[SHOW + ['CountsOfReview', 'review_ratio', 'score']])

reader_engagement()

,Id,Name,Authors,Publisher,Rating,total,CountsOfReview,review_ratio,score
1,24399,Billy Collins Live: A Performance at the Peter Norton Symphony Space,Billy Collins,Random House Audio Publishing Group,4.58,316,57,0.180380,0.942308
2,32270,"Letters From Eden: A Year at Home, in the Woods",Julie Zickefoose,Houghton Mifflin Harcourt,4.52,164,33,0.201220,0.930769
3,88594,Don't Leap with the Sheep: And Other Scriptural Strategies for Avo...,S. Michael Wilcox,Shadow Mountain,4.49,192,35,0.182292,0.925000
4,91614,The Middle Passage: White Ships/ Black Cargo,Tom Feelings,Dial Books,4.46,260,46,0.176923,0.919231
5,11336,Remember: The Journey to School Integration,Toni Morrison,HMH Books for Young Readers,4.41,673,147,0.218425,0.909615
6,40145,Dear Genius: The Letters of Ursula Nordstrom,Ursula Nordstrom,HarperCollins,4.40,570,113,0.198246,0.907692
7,72675,Saturdays and Teacakes,Lester L. Laminack,Peachtree Publishing Company,4.39,407,116,0.285012,0.905769
8,9093,Monkey Portraits,Jill Greenberg,Bulfinch,4.47,272,46,0.169118,0.901231
9,69917,About the Authors: Writing Workshop with Our Youngest Writers,Katie Wood Ray,Heinemann Educational Books,4.48,144,24,0.166667,0.896197
10,9893,Triumph of the Lamb: A Commentary on Revelation,Dennis E. Johnson,P & R Publishing,4.42,182,31,0.170330,0.895056


In [36]:
# Câu 15. Sách theo mức độ tương đồng của phân phối đánh giá (Rating Distribution Similarity)
# Nhập Id; chuẩn hóa 1-5 sao thành tỷ lệ; Cosine; 10 cuốn tương tự.
from sklearn.metrics.pairwise import cosine_similarity

dist = bk[bk['total'] >= 100].copy()
dist_vec = dist[['n1', 'n2', 'n3', 'n4', 'n5']].div(dist[['n1', 'n2', 'n3', 'n4', 'n5']].sum(axis=1), axis=0).values

def rating_dist_similar(book_id, n=10):
    pos_ = np.where(dist['Id'].values == book_id)[0]
    if len(pos_) == 0:
        print('Id không có trong tập sách (cần >= 100 lượt đánh giá).'); return None
    sims = cosine_similarity(dist_vec[pos_[0]].reshape(1, -1), dist_vec)[0]
    out = dist.assign(cosine=sims)
    print('Vector phân phối của sách đầu vào (1-5 sao):', dist_vec[pos_[0]].round(3))
    out = out[out['Id'] != book_id].sort_values(['cosine', 'total'], ascending=False).head(n)
    return rank_table(out[SHOW + ['cosine']])

bid = ask('Nhập mã sách Id', 3, int)
rating_dist_similar(bid)

Nhập mã sách Id [mặc định: 3]: 22
Vector phân phối của sách đầu vào (1-5 sao): [0.03  0.117 0.398 0.287 0.168]


,Id,Name,Authors,Publisher,Rating,total,cosine
1,64432,"On the Planet of Tasteless Pleasure (Bill, The Galactic Hero, #4)",Harry Harrison,Avon Books,3.47,592,0.999782
2,20929,Fantasy Gone Wrong,Martin H. Greenberg,DAW,3.45,238,0.999710
3,13696,"Omnivore (Of Man and Manta, #1)",Piers Anthony,Mundania Press LLC,3.49,1187,0.999691
4,22367,Sebastian O,Grant Morrison,Vertigo,3.42,591,0.999679
5,80772,This Time Love,Elizabeth Lowell,Avon,3.43,901,0.999631
6,41332,"Sorcerer's Moon (Boreal Moon, #3)",Julian May,Ace Books,3.45,365,0.999572
7,28562,"Hard News (Rune, #3)",Jeffery Deaver,Bantam,3.47,1242,0.999509
8,91065,Una Novia Para Un Principe,Rebecca Winters,Harlequin Jazmin,3.50,126,0.999503
9,91066,Bride Fit For A Prince,Rebecca Winters,Harlequin Romance,3.50,126,0.999503
10,82610,The Great Gatenby,John Marsden,Pan Australia,3.44,295,0.999459


In [37]:
# Câu 16. Sách theo năm xuất bản (Publication Period Recommendation)
# Nhập khoảng năm (vd 2000-2015); lọc PublishYear; Top 10 Weighted Rating cao nhất.
def publication_period(y0, y1, n=10):
    r = bk[bk['PublishYear'].between(y0, y1)]
    if r.empty:
        print('Không có sách trong khoảng năm này.'); return None
    return rank_table(r.sort_values('WR', ascending=False).head(n)[SHOW + ['PublishYear', 'WR']])

y0 = ask('Năm xuất bản từ', 2000, int)
y1 = ask('Năm xuất bản đến', 2015, int)
publication_period(y0, y1)

Năm xuất bản từ [mặc định: 2000]: 23
Năm xuất bản đến [mặc định: 2015]: 123
Không có sách trong khoảng năm này.


In [38]:
# Câu 17. Sách dựa trên nhiều tiêu chí sở thích
# Nhập đồng thời tác giả, ngôn ngữ, khoảng số trang, khoảng năm xuất bản, điểm tối thiểu; Top 10.
def multi_criteria(author='', language='', p0=0, p1=10**6, y0=0, y1=3000, min_rating=0.0, n=10):
    r = bk.copy()
    if author:   r = r[r['Authors'].str.contains(author, case=False, na=False, regex=False)]
    if language: r = r[r['Language'].str.lower() == language.lower()]
    r = r[r['pagesNumber'].between(p0, p1) & r['PublishYear'].between(y0, y1) & (r['Rating'] >= min_rating)]
    print('Số sách thỏa tất cả điều kiện:', len(r))
    if r.empty:
        return None
    return rank_table(r.sort_values('WR', ascending=False).head(n)[SHOW + ['Language', 'pagesNumber', 'PublishYear', 'WR']])

a   = ask('Tác giả (để trống = bỏ qua)', '')
lg  = ask('Ngôn ngữ, ví dụ eng, spa, fre (để trống = bỏ qua)', 'eng')
p0  = ask('Số trang tối thiểu', 200, int)
p1  = ask('Số trang tối đa', 500, int)
y0  = ask('Năm xuất bản từ', 2000, int)
y1  = ask('Năm xuất bản đến', 2015, int)
mr  = ask('Điểm đánh giá tối thiểu', 4.0, float)
multi_criteria(a, lg, p0, p1, y0, y1, mr)

Tác giả (để trống = bỏ qua) [mặc định: ]: 
Ngôn ngữ, ví dụ eng, spa, fre (để trống = bỏ qua) [mặc định: eng]: 
Số trang tối thiểu [mặc định: 200]: 
Số trang tối đa [mặc định: 500]: 
Năm xuất bản từ [mặc định: 2000]: 
Năm xuất bản đến [mặc định: 2015]: 
Điểm đánh giá tối thiểu [mặc định: 4.0]: 
Số sách thỏa tất cả điều kiện: 2535


,Id,Name,Authors,Publisher,Rating,total,Language,pagesNumber,PublishYear,WR
1,5,"Harry Potter and the Prisoner of Azkaban (Harry Potter, #3)",J.K. Rowling,Scholastic Inc.,4.57,2610317,eng,435,2004,4.566781
2,15195,The Complete Maus,Art Spiegelman,Penguin Books,4.55,125820,eng,296,2003,4.490741
3,24819,The Calvin And Hobbes: Tenth Anniversary Book,Bill Watterson,Time Warner Books UK,4.63,49824,eng,208,2008,4.482433
4,4255,Harry Potter and the Philosopher's Stone,J.K. Rowling,Bloomsbury Publishing PLC,4.47,6588013,eng,223,2004,4.468910
5,3,"Harry Potter and the Sorcerer's Stone (Harry Potter, #1)",J.K. Rowling,Scholastic Inc,4.47,6587388,eng,309,2003,4.468909
6,15241,"The Two Towers (The Lord of the Rings, #2)",J.R.R. Tolkien,Houghton Mifflin,4.44,675727,eng,322,2003,4.430085
7,4,"Harry Potter and the Chamber of Secrets (Harry Potter, #2)",J.K. Rowling,Scholastic,4.42,2560657,eng,352,2003,4.417445
8,42900,"Lover Awakened (Black Dagger Brotherhood, #3)",J.R. Ward,Signet,4.45,157271,eng,448,2006,4.409063
9,49006,"The Darkest Hour (Warriors, #6)",Erin Hunter,Avon Books,4.53,36557,eng,315,2005,4.367914
10,42389,"Band of Brothers: E Company, 506th Regiment, 101st Airborne from N...",Stephen E. Ambrose,Simon & Schuster; Media Tie-In edition,4.43,91184,eng,432,2002,4.365366


In [39]:
# Câu 20. Đánh giá hệ khuyến nghị sách bằng độ phù hợp tiêu chí và mức đa dạng
# Weighted Rating, Content-Based, Hybrid; so sánh tỷ lệ đáp ứng tiêu chí, đa dạng tác giả, đa dạng NXB, mức trùng lặp.
MIN_RATING, MIN_TOTAL = 4.0, 100
POOL_MIN = 50

def recommend_three(book_id, alpha=0.5, n=10):
    q = bk[bk['Id'] == book_id].iloc[0]
    pool = bk[(bk['Id'] != book_id) & (bk['total'] >= POOL_MIN)].copy()
    pool['sim'] = content_similarity(book_id)[pool.index]
    pool['hybrid'] = alpha * minmax(pool['sim']) + (1 - alpha) * minmax(pool['WR'])
    same_lang = pool[pool['Language'] == q['Language']] if pd.notna(q['Language']) else pool
    return {'Weighted Rating': same_lang.sort_values('WR', ascending=False).head(n),
            'Content-Based':   pool.sort_values(['sim', 'WR'], ascending=False).head(n),
            'Hybrid':          pool.sort_values('hybrid', ascending=False).head(n)}, q

def evaluate_three(book_id, alpha=0.5, n=10):
    recs, q = recommend_three(book_id, alpha, n)
    print('Truy vấn:', q['Name'], '|', q['Authors'])
    rows = {}
    for name, r in recs.items():
        ok = (r['Rating'] >= MIN_RATING) & (r['total'] >= MIN_TOTAL) & (r['Language'] == q['Language'])
        rows[name] = {'Ty_le_dap_ung_tieu_chi': ok.mean(),
                      'Da_dang_tac_gia': r['Authors'].nunique() / len(r),
                      'Da_dang_NXB': r['Publisher'].nunique() / len(r),
                      'Rating_TB': r['Rating'].mean()}
    summary = pd.DataFrame(rows).T
    names = list(recs)
    overlap = pd.DataFrame(index=names, columns=names, dtype=float)
    for a_ in names:
        for b_ in names:
            sa, sb = set(recs[a_]['Id']), set(recs[b_]['Id'])
            overlap.loc[a_, b_] = len(sa & sb) / len(sa | sb)
    return recs, summary, overlap

bid = ask('Nhập mã sách Id (truy vấn)', 3, int)
recs, summary, overlap = evaluate_three(bid)
for name, r in recs.items():
    print(f'\n--- Top 10: {name} ---')
    display(rank_table(r[['Id', 'Name', 'Authors', 'Publisher', 'Rating', 'total']]))
print('\n=== Bảng so sánh ===')
display(summary.round(3))
print('=== Mức trùng lặp giữa các danh sách (Jaccard theo Id) ===')
display(overlap.round(3))

Nhập mã sách Id (truy vấn) [mặc định: 3]: 
Truy vấn: Harry Potter and the Sorcerer's Stone (Harry Potter, #1) | J.K. Rowling

--- Top 10: Weighted Rating ---


,Id,Name,Authors,Publisher,Rating,total
1,71643,"Harry Potter and the Deathly Hallows (Harry Potter, #7)",J.K. Rowling,Arthur A. Levine Books,4.62,2661573
2,8,"Harry Potter Boxed Set, Books 1-5 (Harry Potter, #1-5)",J.K. Rowling,Scholastic,4.78,43968
3,24812,The Complete Calvin and Hobbes,Bill Watterson,Andrews McMeel Publishing,4.82,34079
4,5,"Harry Potter and the Prisoner of Azkaban (Harry Potter, #3)",J.K. Rowling,Scholastic Inc.,4.57,2610317
5,93124,"Harry Potter and the Half-Blood Prince (Harry Potter, #6)",J.K. Rowling,Bloomsbury Publishing,4.57,2306655
6,70367,"Harry Potter and the Half-blood Prince (Harry Potter, #6)",J.K. Rowling,Turtleback Books,4.57,2300327
7,1,"Harry Potter and the Half-Blood Prince (Harry Potter, #6)",J.K. Rowling,Scholastic Inc.,4.57,2298124
8,6,"Harry Potter and the Goblet of Fire (Harry Potter, #4)",J.K. Rowling,Scholastic,4.56,2431085
9,15336,The Lord of the Rings / The Hobbit,J.R.R. Tolkien,Collins Modern Classics,4.60,106636
10,30,J.R.R. Tolkien 4-Book Boxed Set: The Hobbit and The Lord of the Rings,J.R.R. Tolkien,Ballantine Books,4.60,106621



--- Top 10: Content-Based ---


,Id,Name,Authors,Publisher,Rating,total
1,4,"Harry Potter and the Chamber of Secrets (Harry Potter, #2)",J.K. Rowling,Scholastic,4.42,2560657
2,77523,Harry Potter and the Sorcerer's Stone,J.K. Rowling,Arthur A. Levine Books / Scholastic Inc.,4.47,6622480
3,2002,Harry Potter Schoolbooks Box Set: Two Classic Books from the Libra...,J.K. Rowling,Arthur A. Levine,4.40,12706
4,4255,Harry Potter and the Philosopher's Stone,J.K. Rowling,Bloomsbury Publishing PLC,4.47,6588013
5,5,"Harry Potter and the Prisoner of Azkaban (Harry Potter, #3)",J.K. Rowling,Scholastic Inc.,4.57,2610317
6,70357,해리포터와 불사조 기사단 [Part 5 of 5],J.K. Rowling,문학수첩,4.54,67
7,34261,해리포터와 불사조 기사단 [Part 1 of 5],J.K. Rowling,문학수첩,4.31,61
8,49772,"Harri Potter a Maen yr Athronydd (Harry Potter, #1)",J.K. Rowling,Bloomsbury Childrens,4.47,6599670
9,70360,해리포터와 불사조 기사단 [Part 4 of 5],J.K. Rowling,문학수첩,4.36,708
10,72420,"Harry Potter aur Razon ka Kamra (Harry Potter, #2)",J.K. Rowling,"Oxford University Press, USA",4.42,2563515



--- Top 10: Hybrid ---


,Id,Name,Authors,Publisher,Rating,total
1,5,"Harry Potter and the Prisoner of Azkaban (Harry Potter, #3)",J.K. Rowling,Scholastic Inc.,4.57,2610317
2,77523,Harry Potter and the Sorcerer's Stone,J.K. Rowling,Arthur A. Levine Books / Scholastic Inc.,4.47,6622480
3,4255,Harry Potter and the Philosopher's Stone,J.K. Rowling,Bloomsbury Publishing PLC,4.47,6588013
4,4,"Harry Potter and the Chamber of Secrets (Harry Potter, #2)",J.K. Rowling,Scholastic,4.42,2560657
5,49873,"Haris Poteris ir Azkabano kalinys (Haris Poteris, #3)",J.K. Rowling,Alma littera,4.56,2635020
6,93124,"Harry Potter and the Half-Blood Prince (Harry Potter, #6)",J.K. Rowling,Bloomsbury Publishing,4.57,2306655
7,47538,"Harry Potter et le Prisonnier d'Azkaban (Harry Potter, #3)",J.K. Rowling,NaN,4.56,2634925
8,47532,"Harry Potter y el prisionero de Azkaban (Harry Potter, #3)",J.K. Rowling,Salamandra,4.56,2634925
9,49871,"Harry Potter aur Azkaban ka Qaidi (Harry Potter, #3)",J.K. Rowling,"Oxford University Press, USA",4.56,2635020
10,70367,"Harry Potter and the Half-blood Prince (Harry Potter, #6)",J.K. Rowling,Turtleback Books,4.57,2300327



=== Bảng so sánh ===


,Ty_le_dap_ung_tieu_chi,Da_dang_tac_gia,Da_dang_NXB,Rating_TB
Weighted Rating,1.0,0.3,0.8,4.626
Content-Based,0.5,0.1,0.8,4.443
Hybrid,0.6,0.1,0.9,4.531


=== Mức trùng lặp giữa các danh sách (Jaccard theo Id) ===


,Weighted Rating,Content-Based,Hybrid
Weighted Rating,1.000,0.053,0.176
Content-Based,0.053,1.000,0.250
Hybrid,0.176,0.250,1.000
